In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/gas-sensor-array-under-dynamic-gas-mixtures/ethylene_CO.txt
/kaggle/input/gas-sensor-array-under-dynamic-gas-mixtures/ethylene_methane.txt
/kaggle/input/gas-sensor-array-under-dynamic-gas-mixtures/data/ethylene_CO.txt
/kaggle/input/gas-sensor-array-under-dynamic-gas-mixtures/data/ethylene_methane.txt


In [4]:
# ==========================================================
# Setup & Dataset Check
# ==========================================================
import numpy as np
import pandas as pd
import os

# List available files (optional)
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# ==========================================================
# Load the dataset (replace with your correct file)
# ==========================================================
file_path = "/kaggle/input/gas-sensor-array-under-dynamic-gas-mixtures/ethylene_CO.txt"

# Load dataset — skip the first line if it's a messy header
data = pd.read_csv(file_path, sep="\s+", skiprows=1, engine="python")

# Check the column names
print("✅ Columns in the dataset:")
print(data.columns.tolist())


/kaggle/input/gas-sensor-array-under-dynamic-gas-mixtures/ethylene_CO.txt
/kaggle/input/gas-sensor-array-under-dynamic-gas-mixtures/ethylene_methane.txt
/kaggle/input/gas-sensor-array-under-dynamic-gas-mixtures/data/ethylene_CO.txt
/kaggle/input/gas-sensor-array-under-dynamic-gas-mixtures/data/ethylene_methane.txt
✅ Columns in the dataset:
['0.00', '0.00.1', '0.00.2', '-50.85', '-1.95', '-41.82', '1.30', '-4.07', '-28.73', '-13.49', '-3.25', '55139.95', '50669.50', '9626.26', '9762.62', '24544.02', '21420.68', '7650.61', '6928.42']


In [5]:
# ==========================================================
# Decision Tree Model for CO Detection (Ethylene_CO Dataset)
# (Fixed column handling)
# ==========================================================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn import metrics
import joblib

# 1. Load dataset
file_path = "/kaggle/input/gas-sensor-array-under-dynamic-gas-mixtures/ethylene_CO.txt"

# Read and skip messy header lines automatically
data = pd.read_csv(file_path, sep="\s+", skiprows=1, engine="python")

# Now rename columns properly
data.columns = [
    'time_s', 'CO_ppm', 'Ethylene_ppm',
    's1','s2','s3','s4','s5','s6',
    's7','s8','s9','s10','s11','s12','s13','s14','s15','s16'
][:len(data.columns)]  # trims safely if fewer columns

print("✅ Clean shape:", data.shape)
print(data.head())

# 2. Select limited sensors for simplicity (5 sensors)
features = ['s1', 's2', 's3', 's4', 's5']
target = 'CO_ppm'

# 3. Create binary label: 1 = CO present, 0 = no CO
data['CO_present'] = (data[target] > 0).astype(int)

# 4. Split into training and testing
X = data[features]
y = data['CO_present']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Train Decision Tree
model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)

# 6. Evaluate
y_pred = model.predict(X_test)
print("\nAccuracy:", metrics.accuracy_score(y_test, y_pred))
print(metrics.classification_report(y_test, y_pred))

# 7. Display the tree as text
tree_rules = export_text(model, feature_names=features)
print("\n--- Decision Tree Rules ---")
print(tree_rules)

# 8. Save model
joblib.dump(model, "CO_decision_tree.pkl")
print("\nModel saved as CO_decision_tree.pkl")


✅ Clean shape: (4208260, 19)
   time_s  CO_ppm  Ethylene_ppm     s1     s2     s3    s4     s5     s6  \
0    0.01     0.0           0.0 -49.40  -5.53 -42.78  0.49   3.58 -34.55   
1    0.01     0.0           0.0 -40.04 -16.09 -27.59  0.00  -7.16 -42.14   
2    0.03     0.0           0.0 -47.14 -10.57 -32.28  4.40 -11.22 -37.94   
3    0.04     0.0           0.0 -33.58 -20.79 -33.25  6.03   3.42 -34.22   
4    0.05     0.0           0.0 -48.59 -11.54 -36.16  6.03   0.33 -29.05   

      s7    s8        s9       s10      s11      s12       s13       s14  \
0  -9.59  5.37  54395.77  50046.91  9433.20  9591.21  24137.13  20930.33   
1 -12.52 -5.86  53960.02  49299.30  9324.40  9449.81  23628.90  20504.94   
2  -7.16 -1.14  53047.71  48907.00  9170.64  9305.58  23101.66  20101.42   
3 -14.46  8.31  52700.28  48330.96  9073.64  9163.47  22689.54  19694.07   
4 -16.74 -1.14  51910.52  47609.00  8982.88  9021.08  22159.12  19332.57   

       s15      s16  
0  7498.79  6800.66  
1  7369.67  6